# CLOY Processing

This notebook is for you to work with the CLOY data folder.

**Author:** Kelly Chen\
**Date:** 9/29/26

**Table of Contents:**
1. [Goals and Tasks](#sec1)
2. [Vector of Words](#sec2)

<a id="sec1"></a>

## 1. Goals and Tasks

There are two main steps:

1. Turn every CLOY thread (a nested structure of the main post and subsequent comments) into a vector of words (as shown in slides 23-24 of Day 11 notes). [Spend some time looking at the structure of a single thread, since it is nested!]
2. Calculate the similarity score between every two threads  (with cosine similarity) [See sample code below], and save the dataframe into a CSV file.

Additionally:
1. Write a function that given one thread ID, prints out the threadID and subject title of the thread for the top 5 most similar threads.
2. Verification: Use the thread_explorer app to verify that these threads are indeed similar (you can search for the threads and read their content)
3. Remove stop words - Do you think stop words are affecting your results?  What happens if you get rid of them? [TextBlob has a list of stop words.]

<a id="sec2"></a>

## 2. Vector of Words

Import libraries:

In [118]:
import json
from textblob import TextBlob
from textblob import Word
import pandas as pd

from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer #for vector
from sklearn.metrics.pairwise import cosine_similarity 

from nltk.corpus import stopwords
import string

Print threads by reading jsonl file:

In [ ]:
threads = []

file_path = "data/cs315-week4-data/RedditExplorer/cloy_chronological.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip(): 
            data = json.loads(line)
            threads.append(data) 

print(threads[1]) #read first thread to check

{'post_id': 'ffh52z', 'timestamp': 1583693959, 'author': 'bruisedlee123', 'subreddit': 'CrashLandingOnYou', 'title': 'ok, let’s get this started! What are your thoughts about the show?', 'selftext': 'I haven’t seen a kdrama for the longest time and it was actually my dad who chose to stream CLOY on Netflix. He stopped watching after the second episode but I went on after immediately falling in love with the leads. Hyun Bin is just so charismatic as Capt. Ri Jeong Hyeok and Son Ye-jin is perfect as Yoon Se-ri. She’s definitely got the rom-com and drama genre down pat! I also loved Capt. Ri’s squad, most especially Pyo Chi Su ☺️ (we can probably talk about individual characters next time).\n\nCLOY tackled politics, drama and deception, family issues, friendships - and best of all, pure unconditional love, love brought about by fate - all in one highly entertaining and well-written package. \n\nI’m planning on doing a rewatch soon, when I get a chance!', 'url': 'https://www.reddit.com/r/C

Get comments:

In [81]:
comments = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip(): 
            data = json.loads(line)
            comments.append(data["comments"]) 

print(comments[1]) #check comments for first thread

[{'comment_id': 'fk0rooq', 'parent_id': 't3_ffh52z', 'author': 'chimichangarolls', 'comment_body': "i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles... like Seri can cry nonstop but you still know she a boss.", 'score': 19, 'created_utc': 1583764623, 'replies': [{'comment_id': 'fk1c27d', 'parent_id': 't1_fk0rooq', 'author': 'bruisedlee123', 'comment_body': 'Did you have a favorite episode? Mine would have to be ep. 9 ☺️', 'score': 2, 'created_utc': 1583776621, 'replies': [{'comment_id': 'fsfm0lq', 'parent_id': 't1_fk1c27d', 'author': 'tmstms', 'comment_body': "I have not finished it yet, but I already see your point. Episode 9 is when you feel things are going to end happily. Also the stuff in RJH;s parents' home is priceless.", 'score': 2, 'created_utc': 1590941768}]}]}, {'comment_i

This code does not get the comments text for a reply to a main thread. Instead, write a (recursive) function that will get only the main thread comments text and any reply comments. 

In [82]:
def get_comments(comments):
    text = []

    for comment in comments:
        if "comment_body" in comment:
            text.append(comment["comment_body"]) # add this comment's text

        if "replies" in comment:
            text.extend(get_comments(comment["replies"])) # if replies add to comment 

    return text

In [83]:
comments_text = get_comments(comments[1])
print(comments_text) #gets all comments and replies; wrong order however

["i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my favorite part of it other than the couple is that the women have really 3D roles... like Seri can cry nonstop but you still know she a boss.", 'Did you have a favorite episode? Mine would have to be ep. 9 ☺️', "I have not finished it yet, but I already see your point. Episode 9 is when you feel things are going to end happily. Also the stuff in RJH;s parents' home is priceless.", 'I love this show. I stumbled across it on Netflix, gave it a try, and fell in love. I finished it this morning and already miss it. I love Yoon Se-Ri and Jeong Hyeok and nearly all of the other characters. I was so happy with the ending. I’ve added two more kdramas to my Netflix list.', 'Which ones!', 'Itaewon Class, Chocolate, When the Camellia Blooms', 'Thank you :)', "I'm currently obsessed with Hyun Bin. I've never watched Korean dr

Once I have the comments and replies for a thread, I can combine it together with the text from the main thread post.

I will first write a function that gets the text of the thread post written in them and combine it with the comments and replies.

In [132]:
def get_thread_text(thread):
    text = [thread.get("selftext", "")]

    # add comments and replies to main thread text
    text.extend(get_comments(thread.get("comments", [])))

    return " ".join([ch for ch in text if not ch in string.punctuation]) #remove punctuation

Check if the combine function works using the first thread as test:

In [133]:
print(get_thread_text(threads[1]))

I haven’t seen a kdrama for the longest time and it was actually my dad who chose to stream CLOY on Netflix. He stopped watching after the second episode but I went on after immediately falling in love with the leads. Hyun Bin is just so charismatic as Capt. Ri Jeong Hyeok and Son Ye-jin is perfect as Yoon Se-ri. She’s definitely got the rom-com and drama genre down pat! I also loved Capt. Ri’s squad, most especially Pyo Chi Su ☺️ (we can probably talk about individual characters next time).

CLOY tackled politics, drama and deception, family issues, friendships - and best of all, pure unconditional love, love brought about by fate - all in one highly entertaining and well-written package. 

I’m planning on doing a rewatch soon, when I get a chance! i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my favorite part of it other than the couple is that the women have r

Yay, it works! Now compile results of all text into a df for vectorizing:

In [123]:
result = []

for thread in threads: 
    result.append({
        "threadID": thread["post_id"],
        "title": thread["title"], #add title of thread post
        "text": get_thread_text(thread) # all text 
    })

df = pd.DataFrame(result)

In [124]:
df.head()

,threadID,title,text
0,dkxewd,Crash Landing On You has been created,[deleted]
1,ffh52z,"ok, let’s get this started! What are your thou...",I haven’t seen a kdrama for the longest time a...
2,fguorr,Find someone who’ll make you noodles just like...,"😘❤️ Man, those noodles looked so good that I w..."
3,fhcrpl,Captain Ri’s squad! Who’s your favorite?,I can't narrow it down to just one. I love the...
4,fj5byj,Why I think we all need a second season of CLO...,I binged crash landing on you this week and ca...


#### Vectorize into bag-of-words model

Create the vectorizer:

In [125]:
vectorizer = CountVectorizer()

Fit to the text:

In [126]:
X = vectorizer.fit_transform(df["text"])

In [127]:
print(X.shape) #rows, cols or # threads, # unique words

(3075, 30532)


There are 3075 threads and 30532 unique words.

View the words:

In [128]:
words = vectorizer.get_feature_names_out()
print(words)

['00' '000' '0000001' ... '희망도' 'ﾟヮﾟ' '𝓷𝓲𝓬𝓮']


Turn bag of words into df:

In [130]:
words_df = pd.DataFrame(
    X.toarray(),
    columns=words
)

words_df["threadID"] = df["threadID"].values #add thread/post id as new col

print(words_df.head())

   00  000  0000001  0000073127  0000597832  0000709130  0001  0001718292  \
0   0    0        0           0           0           0     0           0   
1   0    0        0           0           0           0     0           0   
2   0    0        0           0           0           0     0           0   
3   0    0        0           0           0           0     0           0   
4   0    0        0           0           0           0     0           0   

   0002833333  0003847248  ...  화이팅  흐음  흔들렸던  흥부와  흥부전  흩날리던  희망도  ﾟヮﾟ  𝓷𝓲𝓬𝓮  \
0           0           0  ...    0   0     0    0    0     0    0    0     0   
1           0           0  ...    0   0     0    0    0     0    0    0     0   
2           0           0  ...    0   0     0    0    0     0    0    0     0   
3           0           0  ...    0   0     0    0    0     0    0    0     0   
4           0           0  ...    0   0     0    0    0     0    0    0     0   

   threadID  
0    dkxewd  
1    ffh52z  
2    fgu

There appears to be numbers mixed in with words. Words also include Korean characters and some words in other special fonts.